In [1]:
from google.colab import files
uploaded = files.upload()


Saving cards.jpg to cards.jpg
Saving flipcard.wav to flipcard.wav
Saving poker.py to poker.py
Saving poker-eval.py to poker-eval.py
Saving poker-hand-testing.csv to poker-hand-testing.csv
Saving poker-hand-training.csv to poker-hand-training.csv


In [2]:
import pandas as pd

df = pd.read_csv("poker-hand-training.csv")
df.head()

,Suit of Card 1,Rank of Card 1,Suit of Card 2,Rank of Card 2,Suit of Card 3,Rank of Card 3,Suit of Card 4,Rank of Card 4,Suit of Card 5,Rank of Card 5,Poker Hand
0,1,10,1,11,1,13,1,12,1,1,9
1,2,11,2,13,2,10,2,12,2,1,9
2,3,12,3,11,3,13,3,10,3,1,9
3,4,10,4,11,4,1,4,13,4,12,9
4,4,1,4,13,4,12,4,11,4,10,9


In [3]:
X = df.iloc[:, :6]
y = df.iloc[:, -1]

In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=500)
model.fit(X, y)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=500)

In [6]:
import pickle

filename = 'poker-model3.sav'
pickle.dump(model, open(filename, 'wb'))

In [7]:
filename = 'poker-model4.sav'
pickle.dump(model, open(filename, 'wb'))

In [8]:
import pickle

model3 = pickle.load(open('poker-model3.sav', 'rb'))
model4 = pickle.load(open('poker-model4.sav', 'rb'))

print("Model 3:", model3.n_features_in_)
print("Model 4:", model4.n_features_in_)

Model 3: 6
Model 4: 6


In [10]:
from sklearn.linear_model import LogisticRegression
import pickle

X = df.iloc[:, :8]
y = df.iloc[:, -1]

model4 = LogisticRegression(max_iter=3000)
model4.fit(X, y)

with open("poker-model4.sav", "wb") as f:
    pickle.dump(model4, f)

print("Model 4:", model4.n_features_in_)

Model 4: 8


In [11]:
import pickle

filename = 'poker-model3.sav'
model3 = pickle.load(open(filename, 'rb'))

filename = 'poker-model4.sav'
model4 = pickle.load(open(filename, 'rb'))

print("Model 3:", model3.n_features_in_)
print("Model 4:", model4.n_features_in_)

Model 3: 6
Model 4: 8


In [13]:
import pandas as pd

def predict_call(hand, opposite_score):
    inp = []
    for suit, rank in hand:
        inp.extend([suit, rank])

    if len(hand) == 3:
        selected_model = model3
    elif len(hand) == 4:
        selected_model = model4
    else:
        raise ValueError("hand must contain 3 or 4 cards")

    X_test = pd.DataFrame(
        [inp],
        columns=selected_model.feature_names_in_
    )

    probabilities = selected_model.predict_proba(X_test)[0]

    expected_score = sum(
        score * probability
        for score, probability in zip(
            selected_model.classes_, probabilities
        )
    )

    return bool(expected_score >= opposite_score)

In [14]:
import numpy as np
import random
import pandas as pd

test = pd.read_csv("poker-hand-testing.csv")

money = [1000] * 10

for i in range(10):
    samp = np.random.randint(0, len(test), size=1000)
    X_test = test.iloc[samp, :-1].to_numpy()
    y_test = test.iloc[samp, -1].to_numpy()

    for test_hand, test_score in zip(X_test, y_test):
        test_input = [
            (suit, rank)
            for suit, rank in zip(test_hand[::2], test_hand[1::2])
        ]

        oppo_score = y_test[random.randrange(len(y_test))]

        if not predict_call(test_input[:3], oppo_score):
            money[i] -= 1
            continue

        if not predict_call(test_input[:4], oppo_score):
            money[i] -= 2
            continue

        if test_score > oppo_score:
            money[i] += 3
        elif test_score < oppo_score:
            money[i] -= 3

    print(f"Test {i + 1}: ${money[i]}")

print("Average money:", round(np.mean(money), 3))

Test 1: $1191
Test 2: $1207
Test 3: $1216
Test 4: $1243
Test 5: $1226
Test 6: $1282
Test 7: $1170
Test 8: $1195
Test 9: $1247
Test 10: $1189
Average money: 1216.6


In [15]:
%%writefile train.py
import pandas as pd
import pickle
from sklearn.linear_model import LogisticRegression

def main():
    df = pd.read_csv("poker-hand-training.csv")
    y = df.iloc[:, -1]

    for cards in (3, 4):
        X = df.iloc[:, :cards * 2]

        model = LogisticRegression(max_iter=3000)
        model.fit(X, y)

        filename = f"poker-model{cards}.sav"
        with open(filename, "wb") as f:
            pickle.dump(model, f)

        print(f"Saved {filename}")

if __name__ == "__main__":
    main()

Writing train.py


In [16]:
from pathlib import Path

user_code = '''
import pickle
import pandas as pd

with open("poker-model3.sav", "rb") as f:
    model3 = pickle.load(f)

with open("poker-model4.sav", "rb") as f:
    model4 = pickle.load(f)

def predict_call(hand, opposite_score):
    inp = []
    for suit, rank in hand:
        inp.extend([suit, rank])

    if len(hand) == 3:
        selected_model = model3
    elif len(hand) == 4:
        selected_model = model4
    else:
        raise ValueError("hand must contain 3 or 4 cards")

    X_test = pd.DataFrame(
        [inp],
        columns=selected_model.feature_names_in_
    )

    probabilities = selected_model.predict_proba(X_test)[0]

    expected_score = sum(
        score * probability
        for score, probability in zip(
            selected_model.classes_, probabilities
        )
    )

    return bool(expected_score >= opposite_score)
'''

path = Path("poker.py")
original = path.read_text(encoding="utf-8")

start_marker = "## ---------------- FILL HERE ---------------------------------"
end_marker = "## -------------------------------------------------"

start = original.index(start_marker) + len(start_marker)
end = original.index(end_marker, start)

updated = original[:start] + "\n" + user_code + "\n" + original[end:]

compile(updated, "poker.py", "exec")

if not Path("poker_original.py").exists():
    Path("poker_original.py").write_text(original, encoding="utf-8")

path.write_text(updated, encoding="utf-8")
print("poker.py updated successfully!")

poker.py updated successfully!


In [17]:
from pathlib import Path

poker_text = Path("poker.py").read_text(encoding="utf-8")

start_marker = "## ---------------- FILL HERE ---------------------------------"
end_marker = "## -------------------------------------------------"

start = poker_text.index(start_marker) + len(start_marker)
end = poker_text.index(end_marker, start)
user_part = poker_text[start:end]

eval_path = Path("poker-eval.py")
eval_text = eval_path.read_text(encoding="utf-8")

header_end = eval_text.index("\n")
second = eval_text.index(end_marker, header_end)

updated = (
    start_marker + "\n"
    + user_part + "\n"
    + eval_text[second:]
)

compile(updated, "poker-eval.py", "exec")
eval_path.write_text(updated, encoding="utf-8")

%run poker-eval.py

average of money= 1239.0


In [18]:
from pathlib import Path
import pickle
import zipfile
from google.colab import files

submission_files = [
    "poker.py",
    "train.py",
    "poker-eval.py",
    "poker-model3.sav",
    "poker-model4.sav",
    "cards.jpg",
    "flipcard.wav",
    "poker-hand-training.csv",
    "poker-hand-testing.csv",
]

for filename in submission_files:
    if not Path(filename).is_file():
        raise FileNotFoundError(f"Missing file: {filename}")

for filename, expected_features in [
    ("poker-model3.sav", 6),
    ("poker-model4.sav", 8),
]:
    with open(filename, "rb") as f:
        saved_model = pickle.load(f)
    assert saved_model.n_features_in_ == expected_features, (
        f"Check model: {filename}"
    )

for filename in ["poker.py", "train.py", "poker-eval.py"]:
    compile(Path(filename).read_text(encoding="utf-8"), filename, "exec")

zip_name = "Project2_Submission.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in submission_files:
        archive.write(filename, arcname=filename)

print("Created:", zip_name)
files.download(zip_name)

Created: Project2_Submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>